In [9]:
import sys
sys.path.append("../src")

import pandas as pd
from openf1_client import get

data = pd.read_csv("../data/processed/dataset_2026.csv")

In [10]:
quali_sessions = get("sessions", use_cache = False, year = 2026, session_name = "Qualifying")

rows = []
for _, s in quali_sessions.iterrows():
    res = get("session_result", session_key = s["session_key"])
    if len(res) > 0:
        res = res[["driver_number", "position"]].copy()
        res["location"] = s["location"]
        rows.append(res)

quali = pd.concat(rows).rename(columns = {"position": "quali_pos"})
data = data.merge(quali, on = ["location", "driver_number"], how = "left")

In [14]:
rows = []

for loc, g in data.groupby("location"):
    rows.append({
        "location": loc,
        "rho_fp2": g["fp2_gap_pct"].corr(g["race_gap_pct"], method = "spearman"),
        "rho_quali": g["quali_pos"].corr(g["race_gap_pct"], method = "spearman"),
    })

scores = pd.DataFrame(rows)
print(scores[["rho_fp2", "rho_quali"]].mean())
print(scores)

rho_fp2      0.777446
rho_quali    0.873001
dtype: float64
        location   rho_fp2  rho_quali
0         Austin  0.847265   0.931889
1      Barcelona  0.818369   0.925697
2       Budapest  0.867647   0.943627
3   Kuala Lumpur  0.841558   0.816883
4         Madrid  0.618182   0.915152
5      Melbourne  0.800000   0.900000
6    Mexico City  0.745098   0.876471
7    Monte Carlo  0.236364   0.763636
8          Monza  0.950464   0.869969
9      Spielberg  0.842105   0.933333
10        Suzuka  0.938235   0.961765
11     São Paulo  0.824060   0.637594


In [12]:
data.to_csv("../data/processed/dataset_2026.csv", index = False)

***Conclusion:*** On the average, qualifying give a better prediction about the race pace, rather than fp2, but as it can be seen at Sao Paulo, Monza and Kuala Lumpur, fp2 perform better. 